In [1]:
# Scraping IMDB Reviews of Barbie

In [2]:
!pip install -q -U openai langchain

In [3]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Open AI API Key:")

Open AI API Key: ········


In [4]:
!pip install -q -U requests

In [5]:
!pip install -q -U scrapy selenium

In [6]:
!apt install chromium-chromedriver

zsh:1: command not found: apt


In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [8]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd

In [10]:
# NO headless this time - open real Chrome so IMDB doesn't block us
options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)


In [11]:
# Load the Barbie reviews page
url = "https://www.imdb.com/title/tt1517268/reviews/?ref_=tt_ov_rt"
driver.get(url)

# Wait for page to fully load
time.sleep(3)

# Check it worked
print(driver.page_source[:300])

<html lang="en-US" xmlns:og="http://opengraphprotocol.org/schema/" xmlns:fb="http://www.facebook.com/2008/fbml"><head><script async="" src="https://images-na.ssl-images-amazon.com/images/I/21auWwkMYgL.js" crossorigin="anonymous"></script><meta charset="utf-8"><meta name="viewport" content="width=dev


In [12]:
page = 1

while page < 50:
    try:
        # Find the new "25 more" button by its class
        load_more = driver.find_element(By.CSS_SELECTOR, 'button.ipc-see-more__button')
        driver.execute_script("arguments[0].click();", load_more)
        time.sleep(2)
        page += 1
        print(f"Loaded page {page}")
    except Exception as e:
        print(f"No more pages to load. Stopped at page {page}. Reason: {e}")
        break

Loaded page 2
Loaded page 3
Loaded page 4
Loaded page 5
Loaded page 6
Loaded page 7
Loaded page 8
Loaded page 9
Loaded page 10
Loaded page 11
Loaded page 12
Loaded page 13
Loaded page 14
Loaded page 15
Loaded page 16
Loaded page 17
Loaded page 18
Loaded page 19
Loaded page 20
Loaded page 21
Loaded page 22
Loaded page 23
Loaded page 24
Loaded page 25
Loaded page 26
Loaded page 27
Loaded page 28
Loaded page 29
Loaded page 30
Loaded page 31
Loaded page 32
Loaded page 33
Loaded page 34
Loaded page 35
Loaded page 36
Loaded page 37
Loaded page 38
Loaded page 39
Loaded page 40
Loaded page 41
Loaded page 42
Loaded page 43
Loaded page 44
Loaded page 45
Loaded page 46
Loaded page 47
Loaded page 48
Loaded page 49
Loaded page 50


In [13]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

# IMDb's new review container class
review_blocks = soup.find_all('article', class_='sc-fa7e37dc-0')

# If that returns 0, try this alternative
if len(review_blocks) == 0:
    review_blocks = soup.find_all('div', attrs={'data-testid': 'review-overflow'})

print(f"Found {len(review_blocks)} reviews")

Found 955 reviews


In [16]:
rating_list = []
review_date_list = []
review_title_list = []
author_list = []
review_list = []
review_url_list = []

for block in tqdm(review_blocks):
    try:
        # Rating
        rating_tag = block.find('span', class_='ipc-rating-star--rating')
        rating = rating_tag.text.strip() if rating_tag else np.nan

        # Review text
        review_tag = block.find('div', class_='ipc-html-content-inner-div')
        review = review_tag.text.strip() if review_tag else np.nan

        # Date
        date_tag = block.find('span', class_='sc-fa7e37dc-7')
        review_date = date_tag.text.strip() if date_tag else np.nan

        # Author
        author_tag = block.find('a', class_='author-span')
        if not author_tag:
            author_tag = block.find('a', attrs={'data-testid': 'author-link'})
        author = author_tag.text.strip() if author_tag else np.nan

        # Title and URL
        title_tag = block.find('a', class_='ipc-title-link-wrapper')
        review_title = title_tag.text.strip() if title_tag else np.nan
        review_url = title_tag['href'] if title_tag else np.nan

        rating_list.append(rating)
        review_date_list.append(review_date)
        review_title_list.append(review_title)
        author_list.append(author)
        review_list.append(review)
        review_url_list.append(review_url)

    except Exception as e:
        print(f"Error: {e}")
        continue

review_df = pd.DataFrame({
    'Review_Date': review_date_list,
    'Author': author_list,
    'Rating': rating_list,
    'Review_Title': review_title_list,
    'Review': review_list,
    'Review_Url': review_url_list
})

print(f"Total reviews extracted: {len(review_df)}")
review_df.head()

100%|██████████████████████| 955/955 [00:00<00:00, 3642.60it/s]

Total reviews extracted: 955


,Review_Date,Author,Rating,Review_Title,Review,Review_Url
0,NaN,NaN,NaN,NaN,"Margot does the best with what she's given, bu...",NaN
1,NaN,NaN,NaN,NaN,Seeing a lot of reviews saying that this movie...,NaN
2,NaN,NaN,NaN,NaN,"I do not usually write reviews, but this is be...",NaN
3,NaN,NaN,NaN,NaN,"The quality, the humor, and the writing of the...",NaN
4,NaN,NaN,NaN,NaN,This is a movie that was way overhyped. Everyo...,NaN


In [17]:
review_df.to_csv("barbie_reviews.csv", index=True)
print("Saved to barbie_reviews.csv ✅")
driver.quit()

Saved to barbie_reviews.csv ✅


In [19]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(
    file_path="barbie_reviews.csv",
    source_column="Review_Url"
)

data = loader.load()
print(f"Loaded {len(data)} documents ✅")

Loaded 955 documents ✅


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)

documents = text_splitter.transform_documents(data)
print(f"Total chunks: {len(documents)}")

Total chunks: 2321


In [34]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

core_embeddings_model = OpenAIEmbeddings()

vector_store = FAISS.from_documents(documents, core_embeddings_model)
print("VectorStore created ✅")

VectorStore created ✅


In [35]:
# Use only first 200 chunks instead of all of them
documents_subset = documents[:200]

vector_store = FAISS.from_documents(documents_subset, core_embeddings_model)
print("VectorStore created ✅")

VectorStore created ✅


In [36]:
query = "How is Will Ferrell in this movie?"
embedding_vector = core_embeddings_model.embed_query(query)
docs = vector_store.similarity_search_by_vector(embedding_vector, k=4)

for page in docs:
    print(page.page_content)
    print("---")

: 35
Review_Date: 
Author: 
Rating: 
Review_Title: 
Review: I really wanted to enjoy this and I know that I am not the target audience but there were massive plot holes and no real flow. The film was very disjointed. Ryan Gosling as good as he is seemed to old to play Ken and Will Ferrell ruined every scene he was in. I just didn't get it, it seemed hollow artificial and hackneyed. A waste of some great talent. It was predictable without being reassuring and trying so hard to be woke in the most superficial way in that but trying to tick so many boxes it actually ticked none. Margo Robbie looks beautiful throughout, the costumes and the sets were amazing but the story was way too weak and didn't make much sense at all.
Review_Url:
---
that is painfully funny.Will Ferrell is essentially playing Lord Business again and him and his whole team of execs are just as cartoonish as the people in Barbie Land but it actually really works as a nice bit of satire that emphasises just how out of to

In [44]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model_name="gpt-3.5-turbo")
retriever = vector_store.as_retriever()

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Returns both the answer AND source documents
qa_with_sources_chain = RunnableParallel(
    answer=(
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    ),
    source_documents=retriever
)

print("QA Chain created ✅")

QA Chain created ✅


In [42]:
response = qa_chain.invoke("Do reviewers consider this movie Kenough?")
print(response)

Based on the reviews provided, it appears that the movie "Kenough" did not meet the expectations of the reviewers. Some felt that it missed the mark in terms of its message and storytelling, while others found it disjointed with significant plot holes. Overall, the majority of reviewers did not consider this movie to be Kenough.


In [45]:
result = qa_with_sources_chain.invoke("How was Will Ferrell in this movie?")
print(result["answer"])
print("---SOURCES---")
for doc in result["source_documents"]:
    print(doc.metadata)

Will Ferrell's performance in the movie was criticized as ruining every scene he was in and playing a similar role to his character in the Lego Movie, which was seen as lazy casting.
---SOURCES---
{'source': '', 'row': 35}
{'source': '', 'row': 78}
{'source': '', 'row': 62}
{'source': '', 'row': 46}


In [46]:
result = qa_with_sources_chain.invoke("Was the movie good for kids?")
print(result["answer"])
print("---SOURCES---")
for doc in result["source_documents"]:
    print(doc.metadata)

Based on the contexts provided, the movie may not be good for kids as it is mentioned that the humor and social commentary may go over their heads, the pacing is described as chaotic and boring at times, and the jokes are not aimed at a younger audience.
---SOURCES---
{'source': '', 'row': 13}
{'source': '', 'row': 63}
{'source': '', 'row': 9}
{'source': '', 'row': 58}


In [47]:
result = qa_with_sources_chain.invoke("What did people think of the movie's message?")
print(result["answer"])
print("---SOURCES---")
for doc in result["source_documents"]:
    print(doc.metadata)

Some people thought the movie's message was balanced and made sense, while others felt it was preachy, biased, and focused too much on hating men.
---SOURCES---
{'source': '', 'row': 43}
{'source': '', 'row': 16}
{'source': '', 'row': 56}
{'source': '', 'row': 26}
